# Train Akira's brains on a free GPU

This notebook trains Akira's own models (Core, Una, Arc, Live, Cast and the router) on a Google Colab GPU,
which is roughly 10-50x faster than a CPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run each cell in order
(Shift+Enter).

Checkpoints are saved to your Google Drive (`MyDrive/akira/models`). If Colab disconnects, which free
sessions do after a few hours, just run all cells again: every brain continues where it stopped.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get Akira's code, and keep checkpoints on Google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

REPO = 'https://github.com/kylanwest11-arch/Akira-.git'
if not os.path.exists('/content/akira'):
    !git clone -q {REPO} /content/akira
else:
    !cd /content/akira && git pull -q
%cd /content/akira
!pip -q install tokenizers pyarrow soundfile scipy

DRIVE_MODELS = '/content/drive/MyDrive/akira/models'
os.makedirs(DRIVE_MODELS, exist_ok=True)
if not os.path.islink('models'):
    !rm -rf models
    os.symlink(DRIVE_MODELS, 'models')
print('checkpoints go to', DRIVE_MODELS)

## Choose what to train

Keep the brains you want in `BRAINS`. The budget is in minutes per brain. Core and the router come first,
because everything else builds on them.

In [ ]:
# 3. Pick brains and minutes (on a T4 GPU)
BRAINS = ['core', 'una', 'arc', 'live', 'cast', 'arc-vision']

JOBS = [
  # name               trainer command                                                                          minutes
  ('core-standard',   'python3 -W ignore -m akira.native.train --brain core-standard --batch 32', 60),
  ('core-pro',        'python3 -W ignore -m akira.native.train --brain core-pro --batch 32', 120),
  ('router-2',        'python3 -W ignore -u scripts/train_router.py --threads 2 --out models/router.pt', 15),
  ('una-lite',        'python3 -W ignore -m akira.native.train --brain una-lite --init models/core-standard.pt --mix code:0.7,chat:0.3 --final_mix code:0.6,chat:0.4 --score code:0.7,chat:0.3', 45),
  ('arc-lite',        'python3 -W ignore -u scripts/train_arc.py --brain arc-lite --batch 128', 90),
  ('live-lite',       'python3 -W ignore -u scripts/train_live.py --brain live-lite', 90),
  ('cast-lite',       'python3 -W ignore -u scripts/train_cast.py --brain cast-lite --batch 96', 60),
  ('arc-vision-lite', 'python3 -W ignore -u scripts/train_arc_vision.py --brain arc-vision-lite --batch 256', 45),
  ('core-ultra',      'python3 -W ignore -m akira.native.train --brain core-ultra --batch 32', 240),
  ('una-general',     'python3 -W ignore -m akira.native.train --brain una-general --init models/core-pro.pt --mix code:0.7,chat:0.3 --final_mix code:0.6,chat:0.4 --score code:0.7,chat:0.3', 60),
  ('una-max',         'python3 -W ignore -m akira.native.train --brain una-max --init models/core-ultra.pt --mix code:0.7,chat:0.3 --final_mix code:0.6,chat:0.4 --score code:0.7,chat:0.3', 90),
  ('arc-general',     'python3 -W ignore -u scripts/train_arc.py --brain arc-general --batch 128', 120),
  ('live-general',    'python3 -W ignore -u scripts/train_live.py --brain live-general', 120),
  ('cast-general',    'python3 -W ignore -u scripts/train_cast.py --brain cast-general --batch 96', 90),
  ('arc-vision-general', 'python3 -W ignore -u scripts/train_arc_vision.py --brain arc-vision-general --batch 256', 60),
]
family = lambda name: 'core' if name.startswith('router') else name.rsplit('-', 1)[0]
jobs = [{'name': n, 'cmd': f'{c} --minutes {m}'} for n, c, m in JOBS if family(n) in BRAINS]
import json
json.dump(jobs, open('models/queue-colab.json', 'w'), indent=1)
print(f'{len(jobs)} jobs, {sum(m for n, c, m in JOBS if family(n) in BRAINS) / 60:.1f} hours of training')

In [ ]:
# 4. Download and prepare the training data (about 20-30 minutes the first time in a session)
brains = ','.join(BRAINS)
!python3 scripts/download_data.py --brains {brains}
if 'core' in BRAINS or 'una' in BRAINS or 'cast' in BRAINS:
    !python3 scripts/build_corpus.py
if 'una' in BRAINS:
    !python3 scripts/build_code_corpus.py
if 'live' in BRAINS:
    !python3 scripts/prepare_live.py
if 'arc-vision' in BRAINS:
    !python3 scripts/prepare_arc_vision.py

In [ ]:
# 5. Train. Progress for each brain is printed as it goes; the full logs are in models/jobs/.
!python3 -u scripts/train_queue.py --queue models/queue-colab.json --once

## Ship the new brains to your Akira server

This copies the trained brains into `release/` (weights only, half size). Then either download the zip and
upload the files to the `release` folder of your GitHub repository, or push straight from here with a GitHub
token (Settings → Developer settings → Personal access tokens, with access to the repository's contents).

In [ ]:
# 6. Export the trained brains
!python3 scripts/export_models.py
!cd release && zip -q -r /content/akira-brains.zip . && ls -la
from google.colab import files
files.download('/content/akira-brains.zip')

In [ ]:
# 7. (Optional) push the new brains to GitHub instead of uploading by hand
from getpass import getpass
token = getpass('GitHub token: ')
!git config user.email "akira@users.noreply.github.com" && git config user.name "Akira trainer"
!git add release && git commit -q -m "Update Akira's brains (trained on Colab)" && \
  git push -q https://{token}@github.com/kylanwest11-arch/Akira-.git HEAD:main && echo pushed